## 0.提取文章

In [2]:
import os
import json
import xml.etree.ElementTree as ET

# 路径：notebook在 notebooks/ 下运行，所以用 ../ 回到项目根目录
xml_dir = "../data/raw/PMC000xxxxxx"
output_path = "../data/processed/dataset.json"

files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
print(f"共发现 {len(files)} 个XML文件")

records = []

for fname in files:
    path = os.path.join(xml_dir, fname)
    try:
        tree = ET.parse(path)
        root = tree.getroot()

        title_el = root.find(".//article-title")
        title = "".join(title_el.itertext()).strip() if title_el is not None else None

        abstract_el = root.find(".//abstract")
        abstract = " ".join(abstract_el.itertext()).strip() if abstract_el is not None else None

        journal_el = root.find(".//journal-meta//journal-title")
        journal = journal_el.text.strip() if journal_el is not None and journal_el.text else None

        pmid_el = root.find('.//article-id[@pub-id-type="pmid"]')
        pmid = pmid_el.text.strip() if pmid_el is not None and pmid_el.text else None

        pub_date = None
        for pub_type in ["ppub", "epub", "pmc-release"]:
            date_el = root.find(f'.//pub-date[@pub-type="{pub_type}"]')
            if date_el is not None:
                year = date_el.findtext("year")
                month = date_el.findtext("month")
                day = date_el.findtext("day")
                if year:
                    pub_date = "-".join(filter(None, [year, month, day]))
                    break

        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "pub_date": pub_date,
        })
    except Exception as e:
        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": None, "title": None, "abstract": None,
            "journal": None, "pub_date": None,
            "_parse_error": str(e),
        })

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)

print(f"提取完成，共 {len(records)} 条记录，已保存到 {output_path}")
print("样例:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

共发现 3028 个XML文件
提取完成，共 3028 条记录，已保存到 ../data/processed/dataset.json
样例:
{
  "pmc_id": "PMC524367",
  "pmid": "15462679",
  "title": "Perceived personal, social and environmental barriers to weight maintenance among young women: A community survey",
  "abstract": "Background Young women are a group at high risk of weight gain. This study examined a range of perceived personal, social and environmental barriers to physical activity and healthy eating for weight maintenance among young women, and how these varied by socioeconomic status (SES), overweight status and domestic situation. Methods In October-December 2001, a total of 445 women aged 18–32 years, selected randomly from the Australian electoral roll, completed a mailed self-report survey that included questions on 11 barriers to physical activity and 11 barriers to healthy eating (relating to personal, social and environmental factors). Height, weight and socio-demographic details were also obtained. Statistical analyses were con

## 1.检查字段完整性

In [10]:
import json

with open("../data/processed/dataset.json", encoding="utf-8") as f:
    data = json.load(f)

n = len(data)
print(f"=== 数据集总量: {n} 条 ===\n")

print("--- 字段缺失率 ---")
fields = ["pmid", "title", "abstract", "journal", "pub_date"]
missing_report = {}
for field in fields:
    missing = sum(1 for r in data if not r.get(field))
    rate = missing / n * 100
    missing_report[field] = rate
    print(f"{field:12s}: 缺失 {missing:4d} 条 / {n} ({rate:.2f}%)")

print()
abstract_missing_rate = missing_report["abstract"]
if abstract_missing_rate > 1:
    print(f"[判断] abstract 缺失率 {abstract_missing_rate:.2f}% > 1%，需要制定清洗策略。")
    print("[策略建议] abstract 是RAG检索的核心内容，缺失时无法靠填充弥补，")
    print("           建议直接丢弃这部分记录。")
else:
    print(f"[判断] abstract 缺失率 {abstract_missing_rate:.2f}% <= 1%，可直接丢弃，对数据量影响很小。")

# 解析失败（XML格式异常）导致的缺失
parse_errors = sum(1 for r in data if r.get("_parse_error"))
print(f"\n其中因XML解析报错导致整条记录缺失: {parse_errors} 条")

=== 数据集总量: 3028 条 ===

--- 字段缺失率 ---
pmid        : 缺失  275 条 / 3028 (9.08%)
title       : 缺失    0 条 / 3028 (0.00%)
abstract    : 缺失  253 条 / 3028 (8.36%)
journal     : 缺失    0 条 / 3028 (0.00%)
pub_date    : 缺失   12 条 / 3028 (0.40%)

[判断] abstract 缺失率 8.36% > 1%，需要制定清洗策略。
[策略建议] abstract 是RAG检索的核心内容，缺失时无法靠填充弥补，
           建议直接丢弃这部分记录。

其中因XML解析报错导致整条记录缺失: 0 条


## 2.基础质量分析

In [13]:
import re

valid = [r for r in data if r.get("abstract")]
print(f"有效记录数（abstract非空）: {len(valid)}\n")

# 超短摘要
short_threshold = 30 
short_abstracts = [r for r in valid if len(r["abstract"].split()) < short_threshold]
print(f"超短摘要 (<{short_threshold} 词): {len(short_abstracts)} 条 ({len(short_abstracts)/len(valid)*100:.2f}%)")
if short_abstracts:
    print(f"  示例: {short_abstracts[0]['abstract'][:150]}")
    print(f"  来源: {short_abstracts[0]['pmc_id']}")

# 编码错误检测
def has_encoding_error(text):
    return bool(re.search(r'[\x80-\x9f]|\ufffd', text))

encoding_errors = [r for r in valid if has_encoding_error(r["abstract"])]
print(f"\n疑似编码错误: {len(encoding_errors)} 条 ({len(encoding_errors)/len(valid)*100:.2f}%)")
if encoding_errors:
    print(f"  示例: {repr(encoding_errors[0]['abstract'][:100])}")
    print(f"  来源: {encoding_errors[0]['pmc_id']}")

有效记录数（abstract非空）: 2775

超短摘要 (<30 词): 200 条 (7.21%)
  示例: A  Drosophila  protein-protein interaction map was constructed using the LexA system, complementing a previous map using the GAL4 system and adding ma
  来源: PMC545799

疑似编码错误: 0 条 (0.00%)


## 3.关键字段分析

In [18]:
from collections import Counter

journals = Counter(r["journal"] for r in valid if r.get("journal"))
print(f"journal 唯一值数量: {len(journals)}")
print("出现最多的5个期刊:", journals.most_common(5))

years = Counter(r["pub_date"][:4] for r in valid if r.get("pub_date"))
print(f"\npub_date 年份跨度: {min(years)} - {max(years)}")
print("各年份文章数(部分，按年份排序前5):", dict(sorted(years.items())[:5]))

pmid_present = sum(1 for r in valid if r.get("pmid"))
print(f"\n有效记录中pmid存在的比例: {pmid_present}/{len(valid)} ({pmid_present/len(valid)*100:.2f}%)")

sample_pmid = next(r["pmid"] for r in valid if r.get("pmid"))
print(f"pmid格式示例: {sample_pmid}，是否纯数字: {sample_pmid.isdigit()}")

journal 唯一值数量: 139
出现最多的5个期刊: [('PLoS Biology', 349), ('BMC Bioinformatics', 161), ('PLoS Medicine', 90), ('BMC Cancer', 88), ('BMC Genomics', 86)]

pub_date 年份跨度: 2003 - 2024
各年份文章数(部分，按年份排序前5): {'2003': 21, '2004': 1850, '2005': 869, '2024': 23}

有效记录中pmid存在的比例: 2735/2775 (98.56%)
pmid格式示例: 15462679，是否纯数字: True


## 4.Token长度分析

In [21]:
try:
    import tiktoken
    print("tiktoken 可用")
except ImportError:
    print("tiktoken 未安装")

try:
    import transformers
    print("transformers 可用，版本:", transformers.__version__)
except ImportError:
    print("transformers 未安装")

tiktoken 未安装


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 可用，版本: 5.12.1


In [24]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

lengths = []
for r in valid:
    full_text = f"{r.get('title') or ''}\n{r['abstract']}"
    n_tokens = len(tokenizer.encode(full_text, truncation=False))
    lengths.append(n_tokens)

lengths.sort()
n = len(lengths)

def percentile(p):
    idx = int(n * p / 100)
    idx = min(idx, n - 1)
    return lengths[idx]

print(f"=== Token长度分布（{n}篇，title+abstract拼接，真实bge tokenizer）===\n")
for p in [5, 25, 50, 75, 90, 95, 99, 100]:
    print(f"  P{p:<3d}: {percentile(p):4d} tokens")

mean_len = sum(lengths) / n
print(f"\n  均值: {mean_len:.0f} tokens")
print(f"  最小: {lengths[0]} / 最大: {lengths[-1]}")

embedding_limit = 512
p95 = percentile(95)
p99 = percentile(99)
over_limit = sum(1 for l in lengths if l > embedding_limit)

print(f"\n=== 与bge-base-en-v1.5上限({embedding_limit} tokens)对比 ===")
print(f"  P95 = {p95} tokens")
print(f"  P99 = {p99} tokens")
print(f"  超过{embedding_limit} tokens的文章占比: {over_limit}/{n} ({over_limit/n*100:.2f}%)")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


=== Token长度分布（2775篇，title+abstract拼接，真实bge tokenizer）===

  P5  :   44 tokens
  P25 :  261 tokens
  P50 :  352 tokens
  P75 :  440 tokens
  P90 :  527 tokens
  P95 :  576 tokens
  P99 :  686 tokens
  P100:  952 tokens

  均值: 343 tokens
  最小: 8 / 最大: 952

=== 与bge-base-en-v1.5上限(512 tokens)对比 ===
  P95 = 576 tokens
  P99 = 686 tokens
  超过512 tokens的文章占比: 328/2775 (11.82%)


## 5.分层抽样

In [29]:
import random
random.seed(1)

def token_len(r):
    full_text = f"{r.get('title') or ''}\n{r['abstract']}"
    return len(tokenizer.encode(full_text, truncation=False))

short_pool = [r for r in valid if token_len(r) < 261]
medium_pool = [r for r in valid if 261 <= token_len(r) < 440]
long_pool = [r for r in valid if token_len(r) >= 440]

print(f"短文本池(<261 tokens): {len(short_pool)} 篇")
print(f"中等文本池(261-440 tokens): {len(medium_pool)} 篇")
print(f"长文本池(>=440 tokens): {len(long_pool)} 篇")

sample_short = random.sample(short_pool, 8)
sample_medium = random.sample(medium_pool, 8)
sample_long = random.sample(long_pool, 8)

print("\n=== 短文本抽样（8篇，全文）===")
for r in sample_short:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

print("\n\n=== 中等文本抽样（8篇，全文）===")
for r in sample_medium:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

print("\n\n=== 长文本抽样（8篇，全文）===")
for r in sample_long:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

短文本池(<261 tokens): 693 篇
中等文本池(261-440 tokens): 1386 篇
长文本池(>=440 tokens): 696 篇

=== 短文本抽样（8篇，全文）===

[PMC549212] In vitro bioassay as a predictor of in vivo response
Background There is a substantial discrepancy between  in vitro  and  in vivo  experiments. The purpose of the present work was development of a theoretical framework to enable improved prediction of  in vivo  response from  in vitro  bioassay results. Results For dose-response curve reaches a plateau  in vitro  we demonstrated that the  in vivo  response has only one maximum. For biphasic patterns of biological response  in vitro  both the bimodal and biphasic  in vivo  responses might be observed. Conclusion As the main result of this work we have demonstrated that  in vivo  responses might be predicted from dose-effect curves measured  in vitro .
--------------------------------------------------------------------------------

[PMC387267] Calcium Dynamics of Cortical Astrocytic Networks In Vivo
Large and long-lasting 

### 高频词分析

In [38]:
from collections import Counter

STOPWORDS = set("""
the a an of and in to with for was were is are this that on at by as from
have not been also could would should may might can will shall must do does
did has had but more than two one all both other after during there most
group however such when into among each who while over only its those
the a an
""".split())

words = []
for r in valid:
    text = (r.get("title") or "") + " " + r["abstract"]
    for w in text.split():
        w = w.strip('.,();:%').lower()
        if w and w not in STOPWORDS and len(w) > 2:
            words.append(w)

freq = Counter(words)
print("=== 高频词 Top20（去停用词）===")
for w, c in freq.most_common(20):
    print(f"  {w:20s} {c}")

=== 高频词 Top20（去停用词）===
  results              2707
  patients             2125
  background           2024
  study                1860
  these                1830
  cells                1600
  methods              1526
  data                 1508
  expression           1491
  between              1404
  gene                 1387
  which                1337
  genes                1296
  analysis             1283
  cell                 1262
  using                1258
  conclusions          1174
  protein              1130
  used                 1108
  health               1087
